In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

import re
import pandas as pd

from sentence_transformers import SentenceTransformer

import numpy as np

c:\Kaggle\Movie\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load data from Kaggle

In [2]:
# Set the path to the file you'd like to load
file_path = "movies.csv"

# Load the latest version
movies = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rishabhkumar2003/the-movie-database-tmdb-comprehensive-dataset",
  file_path,
  pandas_kwargs={
        "engine": "python"
        }
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

C:\Users\USER\AppData\Local\Temp\ipykernel_19500\2575663204.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  movies = kagglehub.load_dataset(


In [3]:
file_path = "cast.csv"

# Load the latest version
cast = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rishabhkumar2003/the-movie-database-tmdb-comprehensive-dataset",
  file_path,
  pandas_kwargs={
        "engine": "python"
        }
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

C:\Users\USER\AppData\Local\Temp\ipykernel_19500\1974523464.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  cast = kagglehub.load_dataset(


In [4]:
file_path = "crew.csv"

# Load the latest version
crew = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rishabhkumar2003/the-movie-database-tmdb-comprehensive-dataset",
  file_path,
  pandas_kwargs={
        "engine": "python"
        }
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

C:\Users\USER\AppData\Local\Temp\ipykernel_19500\2866834677.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  crew = kagglehub.load_dataset(


In [5]:
file_path = "genres.csv"

# Load the latest version
genres = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rishabhkumar2003/the-movie-database-tmdb-comprehensive-dataset",
  file_path,
  pandas_kwargs={
        "engine": "python"
        }
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

C:\Users\USER\AppData\Local\Temp\ipykernel_19500\1534239871.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  genres = kagglehub.load_dataset(


In [6]:
file_path = "reviews.csv"

# Load the latest version
reviews = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rishabhkumar2003/the-movie-database-tmdb-comprehensive-dataset",
  file_path,
  pandas_kwargs={
        "engine": "python"
        }
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

C:\Users\USER\AppData\Local\Temp\ipykernel_19500\3607840220.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  reviews = kagglehub.load_dataset(


# Content-based model

## Text features preparation

In [7]:
numeric_cols = ["vote_average", "runtime", "popularity", "vote_count", "budget", "revenue"]

for col in numeric_cols:
    if col in movies.columns:
        movies[col] = pd.to_numeric(movies[col], errors="coerce")

In [8]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def clean_text(x):
    x = safe_text(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def split_genres(genres_str):
    genres_str = clean_text(genres_str)
    if not genres_str:
        return []
    return [g.strip() for g in genres_str.split(",") if g.strip()]

for col in ["original_title", "overview", "tagline", "genres", "original_language", "release_date"]:
    if col in movies.columns:
        movies[col] = movies[col].apply(clean_text)

cast["name"] = cast["name"].apply(clean_text)
crew["name"] = crew["name"].apply(clean_text)
crew["job"] = crew["job"].apply(clean_text)

movies["release_year"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.year

directors = (
    crew[crew["job"].str.lower() == "director"]
    .groupby("movie_id")["name"]
    .apply(lambda x: list(pd.unique(x)))
    .reset_index()
    .rename(columns={"name": "director_list"})
)

cast_sorted = cast.sort_values(["movie_id", "cast_order"], ascending=[True, True])

top_cast = (
    cast_sorted.groupby("movie_id")["name"]
    .apply(lambda x: list(x.head(5)))
    .reset_index()
    .rename(columns={"name": "top_cast_list"})
)

movies["id"] = pd.to_numeric(movies["id"], errors="coerce")
cast["movie_id"] = pd.to_numeric(cast["movie_id"], errors="coerce")
crew["movie_id"] = pd.to_numeric(crew["movie_id"], errors="coerce")

movie_profiles = movies.merge(directors, how="left", left_on="id", right_on="movie_id")
movie_profiles = movie_profiles.merge(top_cast, how="left", left_on="id", right_on="movie_id")

for col in ["director_list", "top_cast_list"]:
    movie_profiles[col] = movie_profiles[col].apply(lambda x: x if isinstance(x, list) else [])

movie_profiles["genre_list"] = movie_profiles["genres"].apply(split_genres)

movie_profiles.drop(columns=[c for c in ["movie_id_x", "movie_id_y", "movie_id"] if c in movie_profiles.columns],
                    inplace=True, errors="ignore")

movie_profiles

,id,title,original_title,overview,release_date,runtime,budget,revenue,vote_average,vote_count,...,original_language,adult,video,created_at,updated_at,genres,release_year,director_list,top_cast_list,genre_list
0,2.0,Ariel,Ariel,A Finnish man goes to the city to find a job a...,1988-10-21,73.0,0.0,0.0,7.102,368.0,...,fi,0,0,2026-02-13 12:09:05,2026-02-13T17:39:05.734678,"Drama,Comedy,Crime,Romance",1988.0,[Aki Kaurismäki],"[Turo Pajala, Susanna Haavisto, Matti Pellonpä...","[Drama, Comedy, Crime, Romance]"
1,11.0,Star Wars,Star Wars,Princess Leia is captured and held hostage by ...,1977-05-25,121.0,11000000.0,775398007.0,8.202,21922.0,...,en,0,0,2026-02-13 10:49:07,2026-02-13T16:19:07.277805,"Adventure,Action,Science Fiction",1977.0,[George Lucas],"[Mark Hamill, Harrison Ford, Carrie Fisher, Pe...","[Adventure, Action, Science Fiction]"
2,12.0,Finding Nemo,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",2003-05-30,100.0,94000000.0,940335536.0,7.817,20256.0,...,en,0,0,2026-02-13 10:52:41,2026-02-13T16:22:41.330322,"Animation,Family",2003.0,[Andrew Stanton],"[Albert Brooks, Ellen DeGeneres, Alexander Gou...","[Animation, Family]"
3,13.0,Forrest Gump,Forrest Gump,A man with a low IQ has accomplished great thi...,1994-06-23,142.0,55000000.0,677387716.0,8.462,29226.0,...,en,0,0,2026-02-13 10:49:10,2026-02-13T16:19:10.070631,"Drama,Comedy,Romance",1994.0,[Robert Zemeckis],"[Tom Hanks, Robin Wright, Gary Sinise, Sally F...","[Drama, Comedy, Romance]"
4,14.0,American Beauty,American Beauty,"Lester Burnham, a depressed suburban father in...",1999-09-15,122.0,15000000.0,356296601.0,8.002,12792.0,...,en,0,0,2026-02-13 11:14:17,2026-02-13T16:44:17.072753,Drama,1999.0,[Sam Mendes],"[Kevin Spacey, Annette Bening, Thora Birch, We...",[Drama]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9766,1618945.0,Yoh! Bestie,Yoh! Bestie,"Thando is perpetually unlucky in love, but whe...",2026-02-06,95.0,0.0,0.0,6.136,22.0,...,en,0,0,2026-02-13 10:47:02,2026-02-13T16:17:02.176004,"Comedy,Romance",2026.0,[Johnny Barbuzano],"[Katlego Lebogang, Didie Makobane, Kagiso Modu...","[Comedy, Romance]"
9767,1620991.0,The Investigation of Lucy Letby,The Investigation of Lucy Letby,An engaging and thought-provoking look at Lucy...,2026-02-04,95.0,0.0,0.0,6.450,20.0,...,en,0,0,2026-02-13 10:53:57,2026-02-13T16:23:57.411258,"Crime,Documentary",2026.0,[Dominic Sivyer],[Lucy Letby],"[Crime, Documentary]"
9768,1627792.0,Sagaran,Sagaran,As female motorcycle racer Mica and her mechan...,2026-02-06,70.0,0.0,0.0,0.000,0.0,...,tl,0,0,2026-02-13 13:05:36,2026-02-13T18:35:36.168808,Drama,2026.0,[Sid T. Pascua],"[Ashley Lopez, Yda Manzano, Marco Gomez, Ralmo...",[Drama]
9769,1627796.0,Pansamantala,Pansamantala,A story of a drunken dare that pulls best frie...,2026-02-13,0.0,0.0,0.0,0.000,0.0,...,tl,0,0,2026-02-13 11:42:56,2026-02-13T17:12:56.467912,Drama,2026.0,[King Abalos],"[Athena Red, Karen Lopez, Paula Santos, Rhian ...",[Drama]


## Build profile

In [9]:
def build_text_profile(row):
    parts = []

    original_title = row.get("original_title", "")
    genres = row.get("genre_list", [])
    cast_list = row.get("top_cast_list", [])
    tagline = row.get("tagline", "")
    overview = row.get("overview", "")

    if original_title:
        parts.append(f"original title: {original_title}")

    if genres:
        genre_text = ", ".join(genres)
        parts.append(f"genres: {genre_text}")

    if cast_list:
        parts.append(f"cast: {', '.join(cast_list)}")

    if tagline:
        parts.append(f"tagline: {tagline}")

    if overview:
        parts.append(f"overview: {overview}")

    return ". ".join(parts)

movie_profiles["text_profile"] = movie_profiles.apply(build_text_profile, axis=1)
movie_profiles["text_profile"][0]

'original title: Ariel. genres: Drama, Comedy, Crime, Romance. cast: Turo Pajala, Susanna Haavisto, Matti Pellonpää, Eetu Hilkamo, Erkki Pajala. overview: A Finnish man goes to the city to find a job after the mine where he worked is closed and his father commits suicide.'

## Embedding

In [10]:
embed = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(embed)

embed_text = embed_model.encode(
    movie_profiles["text_profile"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embed_text.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11702.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 153/153 [01:05<00:00,  2.35it/s]

(9771, 384)


## Top k based on embedding

In [11]:
movie_profiles = movie_profiles.reset_index(drop=True)

id_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_profiles["id"].tolist())}
idx_to_id = {idx: movie_id for movie_id, idx in id_to_idx.items()}

def candidates_by_movies(movie_ids, top_k=100, weights=None):
    valid_ids = [m for m in movie_ids if m in id_to_idx]
    if len(valid_ids) == 0:
        raise ValueError("No valid movie_ids found")

    idxs = [id_to_idx[m] for m in valid_ids]
    vecs = embed_text[idxs]

    if weights is None:
        weights = np.ones(len(idxs), dtype=np.float32)
    else:
        weights = np.array(weights, dtype=np.float32)
        if len(weights) != len(idxs):
            raise ValueError("weights length must match movie_ids length")

    weights = weights / weights.sum()
    query_vec = np.average(vecs, axis=0, weights=weights)
    query_vec = query_vec / np.linalg.norm(query_vec)

    scores = embed_text @ query_vec

    for idx in idxs:
        scores[idx] = -1.0

    top_idx = np.argsort(-scores)[:top_k]

    result = movie_profiles.iloc[top_idx][["id", "title", "genres", "release_date"]].copy()
    result["retrieval_score"] = scores[top_idx]
    return result.reset_index(drop=True), query_vec

## Rerank features similarity

In [12]:
def jaccard_similarity(set_a, set_b):
    a = set(set_a)
    b = set(set_b)
    if len(a) == 0 and len(b) == 0:
        return 0.0
    union = a | b
    inter = a & b
    return len(inter) / len(union) if len(union) > 0 else 0.0

def numeric_similarity(query_value, cand_value, max_diff):
    if pd.isna(query_value) or pd.isna(cand_value):
        return 0.0
    diff = abs(query_value - cand_value)
    return max(0.0, 1.0 - diff / max_diff)

In [13]:
def build_query_profile(movie_ids, weights=None):
    valid_ids = [m for m in movie_ids if m in id_to_idx]
    idxs = [id_to_idx[m] for m in valid_ids]

    if weights is None:
        weights = np.ones(len(idxs), dtype=np.float32)
    else:
        weights = np.array(weights, dtype=np.float32)

    weights = weights / weights.sum()

    rows = movie_profiles.iloc[idxs]

    query_vec = np.average(embed_text[idxs], axis=0, weights=weights)
    query_vec = query_vec / np.linalg.norm(query_vec)
    
    all_genres = []
    all_directors = []
    all_cast = []

    for _, row in rows.iterrows():
        all_genres.extend(row["genre_list"])
        all_directors.extend(row["director_list"])
        all_cast.extend(row["top_cast_list"])

    query_profile = {
        "movie_ids": valid_ids,
        "query_vec": query_vec,
        "genre_list": pd.Series(all_genres).dropna().unique().tolist(),
        "director_list": pd.Series(all_cast).dropna().unique().tolist(),
        "top_cast_list": pd.Series(all_cast).dropna().unique().tolist(),
        "release_year": np.average(rows["release_year"].fillna(rows["release_year"].median()), weights=weights),
        "runtime": np.average(rows["runtime"].fillna(rows["runtime"].median()), weights=weights),
        "vote_average": np.average(rows["vote_average"].fillna(rows["vote_average"].median()), weights=weights),
        "original_language": rows["original_language"].mode().iloc[0] if rows["original_language"].notna().any() else ""
    }

    return query_profile

In [14]:
def compute_rerank_features(candidate_ids, query_profile):
    rows = movie_profiles[movie_profiles["id"].isin(candidate_ids)].copy()

    cand_idxs = rows.index.to_list()
    cand_vecs = embed_text[cand_idxs]
    text_scores = cand_vecs @ query_profile["query_vec"]

    rows["text_similarity"] = text_scores

    rows["genre_similarity"] = rows["genre_list"].apply(
        lambda x: jaccard_similarity(query_profile["genre_list"], x)
    )

    rows["director_similarity"] = rows["director_list"].apply(
        lambda x: jaccard_similarity(query_profile["director_list"], x)
    )

    rows["cast_similarity"] = rows["top_cast_list"].apply(
        lambda x: jaccard_similarity(query_profile["top_cast_list"], x)
    )

    rows["year_similarity"] = rows["release_year"].apply(
        lambda x: numeric_similarity(query_profile["release_year"], x, max_diff=20)
    )

    rows["runtime_similarity"] = rows["runtime"].apply(
        lambda x: numeric_similarity(query_profile["runtime"], x, max_diff=40)
    )

    rows["rating_similarity"] = rows["vote_average"].apply(
        lambda x: numeric_similarity(query_profile["vote_average"], x, max_diff=3.5)
    )

    rows["language_match"] = rows["original_language"].apply(
        lambda x: 1.0 if x == query_profile["original_language"] and x != "" else 0.0
    )

    return rows

## Weigts

In [15]:
def rerank_candidates(candidate_ids, query_profile, top_n=5):
    rows = compute_rerank_features(candidate_ids, query_profile)

    rows["final_score"] = (
        0.50 * rows["text_similarity"] +
        0.15 * rows["genre_similarity"] +
        0.05 * rows["director_similarity"] +
        0.10 * rows["cast_similarity"] +
        0.03 * rows["year_similarity"] +
        0.15 * rows["rating_similarity"] +
        0.02 * rows["language_match"]
    )

    rows = rows.sort_values("final_score", ascending=False)

    return rows[
        [
            "id", "title", "genres", "release_date",
            "text_similarity", "genre_similarity",
            "director_similarity", "cast_similarity",
            "year_similarity", "rating_similarity",
            "language_match", "final_score"
        ]
    ].head(top_n).reset_index(drop=True)

## Resiluting func for two stage recommendation

In [29]:
def recommend_two_stage_by_movies(movie_ids, top_k_candidates=100, top_n=5, weights=None):
    candidates, _ = candidates_by_movies(
        movie_ids=movie_ids,
        top_k=top_k_candidates,
        weights=weights
    )

    query_profile = build_query_profile(movie_ids, weights=weights)

    result = rerank_candidates(
        candidate_ids=candidates["id"].tolist(),
        query_profile=query_profile,
        top_n=top_n
    )
    return result[['title', 'final_score']]

In [30]:
def normalize_title(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

movie_profiles["title_norm"] = movie_profiles["title"].apply(normalize_title)
movie_profiles["original_title_norm"] = movie_profiles["original_title"].apply(normalize_title)

def get_movie_ids_from_titles(titles):
    movie_ids = []

    for title in titles:
        title_norm = normalize_title(title)

        candidates = movie_profiles[
            (movie_profiles["title_norm"] == title_norm) |
            (movie_profiles["original_title_norm"] == title_norm)
        ].copy()

        if len(candidates) == 0:
            print(f"Фильм не найден: {title}")
            continue

        sort_cols = [c for c in ["popularity", "vote_count", "vote_average"] if c in candidates.columns]
        if sort_cols:
            candidates = candidates.sort_values(sort_cols, ascending=False)

        movie_id = candidates.iloc[0]["id"]
        movie_ids.append(movie_id)

    return movie_ids

In [31]:
titles = ["Inception"]
movie_ids = get_movie_ids_from_titles(titles)

movie_ids

[np.float64(27205.0)]

In [32]:
rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec)

                 title  final_score
0             Face/Off     0.460132
1              Déjà Vu     0.459508
2  Catch Me If You Can     0.459411
3      Minority Report     0.458625
4             Criminal     0.458471


In [33]:
titles = ["Interstellar"]
movie_ids = get_movie_ids_from_titles(titles)

rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec)

        title  final_score
0       Finch     0.542659
1    Spaceman     0.514830
2        Dune     0.512330
3     Arrival     0.488580
4  Prometheus     0.487372


In [34]:
titles = ["The Notebook"]
movie_ids = get_movie_ids_from_titles(titles)

rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec)

               title  final_score
0         Two Lovers     0.565011
1   The Great Gatsby     0.533079
2                Her     0.525515
3      Lovely, Still     0.525447
4  Collateral Beauty     0.517316


In [38]:
titles = ["Titanic"]
movie_ids = get_movie_ids_from_titles(titles)

rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec), movie_ids

              title  final_score
0    Before Sunrise     0.530638
1    Romeo + Juliet     0.513989
2      Little Women     0.511666
3  Beauty from Pain     0.502147
4               Her     0.486270


(None, [np.float64(597.0)])

In [37]:
titles = ["Titanic", 'The Notebook', 'Pride & Prejudice']
movie_ids = get_movie_ids_from_titles(titles)

rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec), movie_ids


               title  final_score
0     Before Sunrise     0.603437
1         Two Lovers     0.591341
2          Atonement     0.586383
3     Romeo + Juliet     0.566047
4  Collateral Beauty     0.562009


(None, [np.float64(597.0), np.float64(11036.0), np.float64(4348.0)])

In [39]:
titles = ["Star Wars", 'Alien', 'The Empire Strikes Back']
movie_ids = get_movie_ids_from_titles(titles)

rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec), movie_ids

                                          title  final_score
0                            Return of the Jedi     0.719969
1                  Star Wars: The Force Awakens     0.645128
2  Star Wars: Episode III - Revenge of the Sith     0.604672
3                      Star Wars: The Last Jedi     0.598610
4              Star Wars: The Rise of Skywalker     0.579765


(None, [np.float64(11.0), np.float64(348.0), np.float64(1891.0)])

In [ ]:
titles = ["Arrival", 'The Martian', '2001: A Space Odyssey']
movie_ids = get_movie_ids_from_titles(titles)

rec = recommend_two_stage_by_movies(
    movie_ids=movie_ids, 
    top_n=5
)
print(rec), movie_ids

          title  final_score
0  Interstellar     0.582452
1       Contact     0.577417
2         Finch     0.568197
3    Prometheus     0.544671
4      Spaceman     0.537248


(None, [np.float64(329865.0), np.float64(286217.0), np.float64(62.0)])